# 5G Network Intelligence — Transfer Results and Error Analysis

Historical forecasting research. **Default Run All performs no training, calibration or holdout replay.**


## Objective and interpretation

Compare all declared horizons on the original beam population. The WAPE target is <10%, not a promise. Reused validation cannot prove unseen-week performance.


## Environment and saved evidence

Input: completed worker outputs or the repository. Output: paths to saved reports. Missing files stay unavailable; they are never replaced with fabricated results.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
from IPython.display import display

# Find report folders in the repo and attached Kaggle outputs.
REPORT_DIRS = [Path('reports'), Path('../reports')]  # Repo root or notebooks/ working directory.
if Path('/kaggle/input').exists():
    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*validation.csv')})

    REPORT_DIRS += sorted({p.parent for p in Path('/kaggle/input').rglob('*manifest.json')})

def report_path(filename):
    for folder in REPORT_DIRS:
        candidate = folder / filename
        if candidate.exists():
            return candidate
    return None


## Single-source weight ablation

Saved metrics only. No retraining or weight fitting occurs here.


In [ ]:
path = report_path('transfer_pilot_validation.csv')
if path:
    frame = pd.read_csv(path)
    pooled = frame[frame.load_tier == 'All'].copy()
    pooled['WAPE (%)'] = 100 * pooled.wape
    display(pooled[['model','horizon','mae','WAPE (%)','n_valid']])
    display(frame.loc[frame.load_tier != 'All', ['model','horizon','load_tier','mae','wape','n_valid']])
else:
    print('Completed results are not available yet.')


### Independent inference verification

A saved CSV is insufficient: checkpoint hashes and history-only CPU predictions must agree with the cloud replay.


In [ ]:
path = report_path('transfer_pilot_evidence.json')
if path:
    display(json.loads(path.read_text(encoding='utf-8')))
else:
    print('Independent verification not completed; no verified-model claim.')


## Per-source and balanced-pool ablation

Saved metrics only. No retraining or weight fitting occurs here.


In [ ]:
path = report_path('multisource_pilot_validation.csv')
if path:
    frame = pd.read_csv(path)
    pooled = frame[frame.load_tier == 'All'].copy()
    pooled['WAPE (%)'] = 100 * pooled.wape
    display(pooled[['model','horizon','mae','WAPE (%)','n_valid']])
    display(frame.loc[frame.load_tier != 'All', ['model','horizon','load_tier','mae','wape','n_valid']])
else:
    print('Completed results are not available yet.')


### Independent inference verification

A saved CSV is insufficient: checkpoint hashes and history-only CPU predictions must agree with the cloud replay.


In [ ]:
path = report_path('multisource_pilot_evidence.json')
if path:
    display(json.loads(path.read_text(encoding='utf-8')))
else:
    print('Independent verification not completed; no verified-model claim.')


## Representation-matched comparison

Positive gain means lower MAE than target-only under the same preprocessing. Day blocks describe consistency; overlapping forecast targets prevent treating them as independent trials.


In [ ]:
path = report_path('transfer_ablation_comparison.csv')
if path:
    display(pd.read_csv(path))
else:
    print('Comparison not attached; no transfer benefit is assumed.')


## Research conclusion

The completed pilot found negative transfer from NR1800 at all tested weights; the balanced pool also regressed at all horizons. NR3500 gains were only 0.005%, 0.051% and 0.206% relative MAE at 1/6/24h versus target-only. These exploratory gains do not establish generalization. WAPE remains about 41–45%, above the 10% target.

Checkpoint hashes and 81 history-only CPU/cloud comparisons passed with zero difference. Separate cloud jobs reproduced both shared cases exactly. Do not promote from reused validation. Original production artifacts and final holdout remain unchanged. A new unseen chronological evaluation requires a separately frozen protocol.
